# Mastering CatBoost
Welcome to the guide on CatBoost.
### Kaggle Dataset:
[Titanic Dataset](https://www.kaggle.com/c/titanic)


### Why and What: Install Package
**WHY:** Required library.
**WHAT:** pip install.


In [ ]:
!pip install catboost -q


### Why and What: Imports
**WHY:** Libraries for data manipulation.
**WHAT:** pandas, numpy, sklearn, matplotlib.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='whitegrid')


## Part 1: Theory Recap
1. Ordered Target Encoding.
2. Symmetric Trees.
3. No Prediction Shift.
4. Categorical handling.
5. Great defaults.


### Why and What: Loading Data
**WHY:** Real world data.
**WHAT:** Titanic.


In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
display(df.head())


### Why and What: Preprocessing
**WHY:** ML models need clean numerical data.
**WHAT:** Fillna, encode, scale.


In [ ]:
df_clean = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())
df_clean['Fare'] = df_clean['Fare'].fillna(df_clean['Fare'].median())
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])
df_clean['Sex'] = df_clean['Sex'].map({'male': 0, 'female': 1})
df_clean = pd.get_dummies(df_clean, columns=['Embarked'], drop_first=True)
X = df_clean.drop('Survived', axis=1).values
y = df_clean['Survived'].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


## Part 2: From Scratch Implementation
### Why and What: Scratch
**WHY:** Demystify the black box.
**WHAT:** Numpy implementation.


In [ ]:
from sklearn.tree import DecisionTreeRegressor
class ScratchModel:
    def __init__(self, n=50, lr=0.1):
        self.n=n; self.lr=lr; self.trees=[]
    def fit(self, X, y):
        X_enc = X.copy()
        for i in range(X.shape[0]):
            X_enc[i,0] = np.mean(y) if i==0 else (np.sum(y[:i]) + np.mean(y)) / (i+1)
        pred = np.zeros(y.shape[0])
        for _ in range(self.n):
            p = 1 / (1 + np.exp(-pred))
            tree = DecisionTreeRegressor(max_depth=1).fit(X_enc, y - p)
            pred += self.lr * tree.predict(X_enc)
            self.trees.append(tree)
    def predict_proba(self, X):
        X_enc = X.copy(); X_enc[:,0] = 0.5
        pred = np.zeros(X.shape[0])
        for t in self.trees: pred += self.lr * t.predict(X_enc)
        return 1 / (1 + np.exp(-pred))
    def predict(self, X): return (self.predict_proba(X) >= 0.5).astype(int)


### Why and What: Evaluation
**WHY:** Verify correctness.
**WHAT:** Predict and accuracy.


In [ ]:
scratch_model = ScratchModel()
scratch_model.fit(X_train, y_train)
y_pred_custom = scratch_model.predict(X_test)
print(f"Scratch Accuracy: {accuracy_score(y_test, y_pred_custom)*100:.2f}%")


## Part 3: Library Implementation
### Why and What: Library
**WHY:** Optimized production code.
**WHAT:** Official package.


In [ ]:
from catboost import CatBoostClassifier
model_library = CatBoostClassifier(n_estimators=50, verbose=0, random_state=42)
model_library.fit(X_train, y_train)
print('CatBoost Accuracy:', accuracy_score(y_test, model_library.predict(X_test)))


### Why and What: Visualizations
**WHY:** Diagnose behavior.
**WHAT:** ROC and Importances.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if hasattr(model_library, 'predict_proba'):
    fpr, tpr, _ = roc_curve(y_test, model_library.predict_proba(X_test)[:, 1])
    axes[0].plot(fpr, tpr, color='darkorange', label=f'ROC (area = {auc(fpr, tpr):.2f})')
axes[0].plot([0,1], [0,1], 'k--', label='Random Guess')
axes[0].set_title('ROC Curve')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(loc='lower right')
importances = getattr(model_library, 'feature_importances_', None)
if importances is not None:
    idx = np.argsort(importances)
    axes[1].barh(range(len(idx)), importances[idx], color='teal', label='Importance')
    axes[1].set_yticks(range(len(idx)))
    axes[1].set_yticklabels(df_clean.drop('Survived', axis=1).columns[idx])
    axes[1].legend(loc='lower right')
axes[1].set_title('Feature Importances')
axes[1].set_xlabel('Relative Importance'); axes[1].set_ylabel('Features')
plt.tight_layout()
plt.show()


## Part 4: Hyperparameter Experiments
### Why and What: Tuning
**WHY:** Optimize model.
**WHAT:** Grid search.


In [ ]:
res=[]
for lr in [0.01, 0.1]:
 for depth in [3, 5, 7]:
  clf=CatBoostClassifier(depth=depth, learning_rate=lr, iterations=50, verbose=0)
  res.append({'depth':depth, 'lr':lr, 'acc':cross_val_score(clf, X_scaled, y, cv=3).mean()})
sns.lineplot(data=pd.DataFrame(res), x='depth', y='acc', hue='lr')
plt.title('CatBoost Hyperparameters'); plt.legend(title='LR'); plt.show()


## Part 5: Interview Corner
**Q: Why symmetric trees?**
**A:** They use the same split across the entire level of the tree, which prevents overfitting and makes prediction incredibly fast.


## Key Takeaways
- Target Encoding.
- Symmetric Trees.
- Prevents Target Leakage.
- Handles categories well.
- Fast inference.
